In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [13]:
# Define commonly used project folders
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES = PROJECT_ROOT / "figures"
RESULTS = PROJECT_ROOT / "results"

In [5]:
print("Project root:", PROJECT_ROOT)
print("Raw data path:", DATA_RAW)

Project root: D:\multi-horizon-asset-price-forecasting
Raw data path: D:\multi-horizon-asset-price-forecasting\data\raw


In [7]:
list(DATA_RAW.iterdir())

[WindowsPath('D:/multi-horizon-asset-price-forecasting/data/raw/.gitkeep'),
 WindowsPath('D:/multi-horizon-asset-price-forecasting/data/raw/BTCUSD.csv'),
 WindowsPath('D:/multi-horizon-asset-price-forecasting/data/raw/ETHUSD..csv'),
 WindowsPath('D:/multi-horizon-asset-price-forecasting/data/raw/GOOGL.csv'),
 WindowsPath('D:/multi-horizon-asset-price-forecasting/data/raw/QQQ.csv'),
 WindowsPath('D:/multi-horizon-asset-price-forecasting/data/raw/TSLA.csv')]

In [8]:
tsla=pd.read_csv(DATA_RAW/'TSLA.CSV')

In [9]:
tsla.head()

,Date,Open,High,Low,Close,Volume
0,2010-06-28,1.13333,1.13333,1.13333,1.13333,0
1,2010-06-29,1.26667,1.66667,1.16933,1.59267,281749140
2,2010-06-30,1.71933,2.02800,1.55333,1.58867,257915910
3,2010-07-01,1.66667,1.72800,1.35133,1.46400,123447945
4,2010-07-02,1.53333,1.54000,1.24733,1.28000,77127105


In [11]:
tsla.info()

<class 'pandas.DataFrame'>
RangeIndex: 4064 entries, 0 to 4063
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    4064 non-null   str    
 1   Open    4064 non-null   float64
 2   High    4064 non-null   float64
 3   Low     4064 non-null   float64
 4   Close   4064 non-null   float64
 5   Volume  4064 non-null   int64  
dtypes: float64(4), int64(1), str(1)
memory usage: 190.6 KB


In [12]:
tsla.tail()

,Date,Open,High,Low,Close,Volume
4059,2026-08-19,338.89,351.62,335.7000,351.12,36735428
4060,2026-08-20,346.20,347.50,338.9600,345.13,30766360
4061,2026-08-21,349.88,366.50,346.9000,362.86,59223958
4062,2026-08-24,361.41,363.24,348.2600,348.95,39190484
4063,2026-08-25,349.58,357.00,349.2003,350.25,29816685


In [14]:
tsla['Date']=pd.to_datetime(tsla['Date'])

In [15]:
tsla.info()

<class 'pandas.DataFrame'>
RangeIndex: 4064 entries, 0 to 4063
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    4064 non-null   datetime64[us]
 1   Open    4064 non-null   float64       
 2   High    4064 non-null   float64       
 3   Low     4064 non-null   float64       
 4   Close   4064 non-null   float64       
 5   Volume  4064 non-null   int64         
dtypes: datetime64[us](1), float64(4), int64(1)
memory usage: 190.6 KB


In [16]:
tsla.isnull().sum()

Date      0
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64

In [17]:
tsla.duplicated().sum()

np.int64(0)

In [18]:
tsla['Date'].duplicated().sum()

np.int64(0)

In [19]:
invalid_ohlc = tsla[
    (tsla["High"] < tsla["Low"]) |
    (tsla["High"] < tsla["Open"]) |
    (tsla["High"] < tsla["Close"]) |
    (tsla["Low"] > tsla["Open"]) |
    (tsla["Low"] > tsla["Close"])
]

invalid_ohlc

,Date,Open,High,Low,Close,Volume


In [27]:
(tsla[["Open", "High", "Low", "Close"]] <= 0).sum()


Open     0
High     0
Low      0
Close    0
dtype: int64

In [28]:
(tsla["Volume"] < 0).sum()

np.int64(0)

In [29]:
(tsla["Volume"] == 0).sum()

np.int64(1)

In [30]:
tsla=tsla.set_index('Date')

In [32]:
tsla.to_csv(DATA_PROCESSED/'TSLA_processed.csv')

In [33]:
list(DATA_PROCESSED.iterdir())

[WindowsPath('D:/multi-horizon-asset-price-forecasting/data/processed/.gitkeep'),
 WindowsPath('D:/multi-horizon-asset-price-forecasting/data/processed/TSLA_processed.csv')]

In [36]:
def clean_stooq_data(df):
    """
    Clean and validate Stooq historical price data.

    Required columns:
    Date, Open, High, Low, Close

    Optional column:
    Volume
    """

    # 1. Make a copy to avoid modifying the original dataframe
    cleaned_df = df.copy()

    # 2. Convert Date column to datetime
    cleaned_df["Date"] = pd.to_datetime(cleaned_df["Date"])

    # 3. Sort data from oldest to newest
    cleaned_df = cleaned_df.sort_values("Date")

    # 4. Remove duplicate rows
    cleaned_df = cleaned_df.drop_duplicates()

    # 5. Remove duplicate dates
    cleaned_df = cleaned_df.drop_duplicates(subset="Date")

    # 6. Check missing values
    print("Missing values:")
    print(cleaned_df.isnull().sum())

    # 7. Check OHLC consistency
    invalid_ohlc = cleaned_df[
        (cleaned_df["High"] < cleaned_df["Low"]) |
        (cleaned_df["High"] < cleaned_df["Open"]) |
        (cleaned_df["High"] < cleaned_df["Close"]) |
        (cleaned_df["Low"] > cleaned_df["Open"]) |
        (cleaned_df["Low"] > cleaned_df["Close"])
    ]

    print("\nInvalid OHLC rows:", len(invalid_ohlc))

    # 8. Check non-positive prices
    non_positive_prices = (
        cleaned_df[["Open", "High", "Low", "Close"]] <= 0
    ).sum()

    print("\nNon-positive prices:")
    print(non_positive_prices)

    # 9. Check Volume only if the column exists
    if "Volume" in cleaned_df.columns:
        print("\nNegative volume:", (cleaned_df["Volume"] < 0).sum())
        print("Zero volume:", (cleaned_df["Volume"] == 0).sum())
    else:
        print("\nVolume column not available.")

    # 10. Set Date as index
    cleaned_df = cleaned_df.set_index("Date")

    return cleaned_df

In [39]:
googl = pd.read_csv(DATA_RAW / "GOOGL.csv")
qqq = pd.read_csv(DATA_RAW / "QQQ.csv")
btcusd = pd.read_csv(DATA_RAW / "BTCUSD.csv")

In [41]:
googl_clean = clean_stooq_data(googl)
qqq_clean = clean_stooq_data(qqq)
btcusd_clean = clean_stooq_data(btcusd)

Missing values:
Date      0
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64

Invalid OHLC rows: 0

Non-positive prices:
Open     0
High     0
Low      0
Close    0
dtype: int64

Negative volume: 0
Zero volume: 1
Missing values:
Date      0
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64

Invalid OHLC rows: 0

Non-positive prices:
Open     0
High     0
Low      0
Close    0
dtype: int64

Negative volume: 0
Zero volume: 0
Missing values:
Date     0
Open     0
High     0
Low      0
Close    0
dtype: int64

Invalid OHLC rows: 0

Non-positive prices:
Open     0
High     0
Low      0
Close    0
dtype: int64

Volume column not available.


In [42]:
googl_clean.to_csv(DATA_PROCESSED/'GOOGL_processes.csv')

In [43]:
qqq_clean.to_csv(DATA_PROCESSED / "QQQ_processed.csv")
btcusd_clean.to_csv(DATA_PROCESSED / "BTCUSD_processed.csv")


In [48]:
list(DATA_PROCESSED.iterdir())

[WindowsPath('D:/multi-horizon-asset-price-forecasting/data/processed/.gitkeep'),
 WindowsPath('D:/multi-horizon-asset-price-forecasting/data/processed/BTCUSD_processed.csv'),
 WindowsPath('D:/multi-horizon-asset-price-forecasting/data/processed/GOOGL_processes.csv'),
 WindowsPath('D:/multi-horizon-asset-price-forecasting/data/processed/QQQ_processed.csv'),
 WindowsPath('D:/multi-horizon-asset-price-forecasting/data/processed/TSLA_processed.csv')]

In [57]:
eth=pd.read_csv(DATA_RAW/'ETHUSD.csv',sep=';')

In [58]:
eth.head()

,timeOpen,timeClose,timeHigh,timeLow,name,open,high,low,close,volume,marketCap,circulatingSupply,timestamp
0,2026-08-25T00:00:00.000Z,2026-08-25T23:59:59.999Z,2026-08-25T02:31:00.000Z,2026-08-25T21:09:00.000Z,2781,2481.777745,2531.084546,2416.602717,2443.159467,1.727086e+10,2.947628e+11,1.206813e+08,2026-08-25T23:59:59.999Z
1,2026-08-24T00:00:00.000Z,2026-08-24T23:59:59.999Z,2026-08-24T15:28:00.000Z,2026-08-24T02:03:00.000Z,2781,2463.640662,2531.744821,2425.387936,2481.831280,2.285223e+10,2.995047e+11,1.206815e+08,2026-08-24T23:59:59.999Z
2,2026-08-23T00:00:00.000Z,2026-08-23T23:59:59.999Z,2026-08-23T21:35:00.000Z,2026-08-23T05:18:00.000Z,2781,2424.217124,2483.930367,2357.139895,2463.815947,1.826925e+10,2.973159e+11,1.206815e+08,2026-08-23T23:59:59.999Z
3,2026-08-22T00:00:00.000Z,2026-08-22T23:59:59.999Z,2026-08-22T04:43:00.000Z,2026-08-22T10:24:00.000Z,2781,2515.392280,2528.896640,2389.902008,2424.251232,2.398529e+10,2.925583e+11,1.206816e+08,2026-08-22T23:59:59.999Z
4,2026-08-21T00:00:00.000Z,2026-08-21T23:59:59.999Z,2026-08-21T22:10:00.000Z,2026-08-21T00:05:00.000Z,2781,2326.382226,2545.881089,2324.709573,2515.281795,3.332857e+10,3.035618e+11,1.206817e+08,2026-08-21T23:59:59.999Z


In [59]:
eth.info()

<class 'pandas.DataFrame'>
RangeIndex: 4037 entries, 0 to 4036
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   timeOpen           4037 non-null   str    
 1   timeClose          4037 non-null   str    
 2   timeHigh           4037 non-null   str    
 3   timeLow            4037 non-null   str    
 4   name               4037 non-null   int64  
 5   open               4037 non-null   float64
 6   high               4037 non-null   float64
 7   low                4037 non-null   float64
 8   close              4037 non-null   float64
 9   volume             4037 non-null   float64
 10  marketCap          4037 non-null   float64
 11  circulatingSupply  4037 non-null   float64
 12  timestamp          4037 non-null   str    
dtypes: float64(7), int64(1), str(5)
memory usage: 410.1 KB


In [60]:
eth["Date"] = pd.to_datetime(eth["timestamp"])

eth = eth[
    ["Date", "open", "high", "low", "close", "volume"]
].copy()

eth.columns = [
    "Date", "Open", "High", "Low", "Close", "Volume"
]

In [61]:
eth.head()

,Date,Open,High,Low,Close,Volume
0,2026-08-25 23:59:59.999000+00:00,2481.777745,2531.084546,2416.602717,2443.159467,1.727086e+10
1,2026-08-24 23:59:59.999000+00:00,2463.640662,2531.744821,2425.387936,2481.831280,2.285223e+10
2,2026-08-23 23:59:59.999000+00:00,2424.217124,2483.930367,2357.139895,2463.815947,1.826925e+10
3,2026-08-22 23:59:59.999000+00:00,2515.392280,2528.896640,2389.902008,2424.251232,2.398529e+10
4,2026-08-21 23:59:59.999000+00:00,2326.382226,2545.881089,2324.709573,2515.281795,3.332857e+10


In [62]:
eth.info()

<class 'pandas.DataFrame'>
RangeIndex: 4037 entries, 0 to 4036
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype              
---  ------  --------------  -----              
 0   Date    4037 non-null   datetime64[us, UTC]
 1   Open    4037 non-null   float64            
 2   High    4037 non-null   float64            
 3   Low     4037 non-null   float64            
 4   Close   4037 non-null   float64            
 5   Volume  4037 non-null   float64            
dtypes: datetime64[us, UTC](1), float64(5)
memory usage: 189.4 KB


In [67]:
eth["Date"] = pd.to_datetime(eth["Date"]).dt.date

In [68]:
eth=eth.sort_values('Date')

In [69]:
eth.head()

,Date,Open,High,Low,Close,Volume
4036,2015-08-07,2.831620,3.536610,2.521120,2.772120,164329.0
4035,2015-08-08,2.793760,2.798810,0.714725,0.753325,674188.0
4034,2015-08-09,0.706136,0.879810,0.629191,0.701897,532170.0
4033,2015-08-10,0.713989,0.729854,0.636546,0.708448,405283.0
4032,2015-08-11,0.708087,1.131410,0.663235,1.067860,1463100.0


In [70]:
eth.info()

<class 'pandas.DataFrame'>
RangeIndex: 4037 entries, 4036 to 0
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    4037 non-null   object 
 1   Open    4037 non-null   float64
 2   High    4037 non-null   float64
 3   Low     4037 non-null   float64
 4   Close   4037 non-null   float64
 5   Volume  4037 non-null   float64
dtypes: float64(5), object(1)
memory usage: 189.4+ KB


In [71]:
eth=eth.set_index('Date')

In [72]:
eth.head()

,Open,High,Low,Close,Volume
Date,,,,,
2015-08-07,2.831620,3.536610,2.521120,2.772120,164329.0
2015-08-08,2.793760,2.798810,0.714725,0.753325,674188.0
2015-08-09,0.706136,0.879810,0.629191,0.701897,532170.0
2015-08-10,0.713989,0.729854,0.636546,0.708448,405283.0
2015-08-11,0.708087,1.131410,0.663235,1.067860,1463100.0


In [73]:
eth.tail()

,Open,High,Low,Close,Volume
Date,,,,,
2026-08-21,2326.382226,2545.881089,2324.709573,2515.281795,3.332857e+10
2026-08-22,2515.392280,2528.896640,2389.902008,2424.251232,2.398529e+10
2026-08-23,2424.217124,2483.930367,2357.139895,2463.815947,1.826925e+10
2026-08-24,2463.640662,2531.744821,2425.387936,2481.831280,2.285223e+10
2026-08-25,2481.777745,2531.084546,2416.602717,2443.159467,1.727086e+10


In [75]:
eth.to_csv(DATA_PROCESSED/'ETHUSD_processes.csv')

In [76]:
list(DATA_PROCESSED.iterdir())

[WindowsPath('D:/multi-horizon-asset-price-forecasting/data/processed/.gitkeep'),
 WindowsPath('D:/multi-horizon-asset-price-forecasting/data/processed/BTCUSD_processed.csv'),
 WindowsPath('D:/multi-horizon-asset-price-forecasting/data/processed/ETHUSD_processes.csv'),
 WindowsPath('D:/multi-horizon-asset-price-forecasting/data/processed/GOOGL_processes.csv'),
 WindowsPath('D:/multi-horizon-asset-price-forecasting/data/processed/QQQ_processed.csv'),
 WindowsPath('D:/multi-horizon-asset-price-forecasting/data/processed/TSLA_processed.csv')]